# Module 2: RAG Pattern Implementation

This notebook implements a conversational AI assistant for Airbnb listings using:
- **Retrieval-Augmented Generation (RAG)** pattern
- **LangChain** for building the RAG pipeline
- **Custom DocumentDB Retriever** for vector search
- **Conversation Memory** for context-aware responses

## Learning Objectives
- Understand the RAG pattern and its components
- Build a custom retriever using vector search results
- Implement conversational AI with LangChain
- Create context-aware prompts for better AI responses
- Handle conversation memory and follow-up questions

## Step 1: Import Required Libraries and Setup Environment

In [ ]:
import os
from typing import List, Dict, Tuple
from pymongo import MongoClient
from openai import OpenAI
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Load environment variables
load_dotenv(override=True)

print("✅ Libraries imported and environment loaded")

## Step 2: Initialize Connections and LLM

In [ ]:
# Initialize OpenAI client
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Connect to DocumentDB
DOCUMENTDB_CONNECTION_STRING = os.getenv('DOCUMENTDB_CONNECTION_STRING')
mongo_client = MongoClient(DOCUMENTDB_CONNECTION_STRING)
db = mongo_client['db']
collection = db['listings']

# Initialize LangChain LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.7,  # Balance between creativity and consistency
    api_key=os.getenv('OPENAI_API_KEY')
)

print("✅ Connections initialized")
print(f"📊 Database: {collection.count_documents({})} listings available")

In [ ]:
# Verify connection by fetching one document
test_doc = collection.find_one()
if test_doc:
    print(f"✅ Successfully retrieved document: {test_doc.get('name', 'Unknown')}")
    print(f"📊 Has embedding: {'descriptionVector' in test_doc}")
else:
    print("⚠️ No documents found. Please run Module 1 first to load data.")

## Step 3: Understanding RAG Architecture

RAG (Retrieval-Augmented Generation) combines:
1. **Retrieval**: Finding relevant information from a knowledge base (our vector search)
2. **Augmentation**: Adding that information to the AI's context
3. **Generation**: Using an LLM to generate responses based on the retrieved context

### Why RAG?
- ✅ AI has access to your current, specific data
- ✅ Responses are grounded in real information
- ✅ Can cite sources and provide accurate details
- ✅ Knowledge base is updateable without retraining

## Step 4: Create Embedding Generation Function

In [ ]:
def generate_embedding(text: str) -> List[float]:
    """
    Generate a vector embedding for the given text using OpenAI.
    
    Args:
        text (str): The text to embed
        
    Returns:
        list: A 1536-dimension vector representing the text
    """
    if not text or not isinstance(text, str):
        return None
    
    try:
        response = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Error generating embedding: {e}")
        return None

# Test the function
test_embedding = generate_embedding("cozy apartment near downtown")
print(f"✅ Embedding function ready")
print(f"📏 Test embedding dimensions: {len(test_embedding)}")

## Step 5: Create a Custom DocumentDB Retriever

LangChain uses "retrievers" to fetch relevant documents. We'll create a custom retriever that uses our DocumentDB vector search.

In [ ]:
class CustomDocumentDBRetriever(BaseRetriever):
    """
    Custom retriever that uses DocumentDB vector search to find relevant listings.
    """
    
    collection: any = None
    openai_client: any = None
    top_k: int = 5
    
    class Config:
        arbitrary_types_allowed = True
    
    def _generate_embedding(self, text: str) -> List[float]:
        """Generate embedding for the given text."""
        response = self.openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding
    
    def _get_relevant_documents(self, query: str) -> List[Document]:
        """
        Retrieve relevant documents using vector search.
        
        Args:
            query: The search query
            
        Returns:
            List of Document objects with content and metadata
        """
        # Generate embedding for the query
        query_embedding = self._generate_embedding(query)
        
        # Perform vector search
        pipeline = [
            {
                "$search": {
                    "cosmosSearch": {
                        "vector": query_embedding,
                        "path": "descriptionVector",
                        "k": self.top_k
                    },
                    "returnStoredSource": True
                }
            },
            {
                "$project": {
                    "_id": 1,
                    "name": 1,
                    "description": 1,
                    "summary": 1,
                    "property_type": 1,
                    "bedrooms": 1,
                    "beds": 1,
                    "price": 1,
                    "address": 1,
                    "amenities": 1,
                    "searchScore": {"$meta": "searchScore"}
                }
            }
        ]
        
        results = list(self.collection.aggregate(pipeline))
        
        # Convert to LangChain Document format
        documents = []
        for result in results:
            # Create rich content for the LLM
            content = f"""
Property: {result.get('name', 'N/A')}
Type: {result.get('property_type', 'N/A')}
Location: {result.get('address', {}).get('market', 'N/A')}, {result.get('address', {}).get('country', 'N/A')}
Bedrooms: {result.get('bedrooms', 'N/A')} | Beds: {result.get('beds', 'N/A')}
Price: ${result.get('price', 'N/A')} per night
Amenities: {', '.join(result.get('amenities', [])[:10])}
Description: {result.get('description', result.get('summary', 'N/A'))[:500]}
"""
            
            # Create Document with metadata
            doc = Document(
                page_content=content,
                metadata={
                    "id": str(result.get('_id', '')),
                    "name": result.get('name', ''),
                    "price": result.get('price', 0),
                    "property_type": result.get('property_type', ''),
                    "similarity_score": result.get('searchScore', 0)
                }
            )
            documents.append(doc)
        
        return documents
    
    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        """Async version of _get_relevant_documents."""
        return self._get_relevant_documents(query)

print("✅ CustomDocumentDBRetriever class defined")

In [ ]:
# Initialize the retriever
retriever = CustomDocumentDBRetriever(
    collection=collection,
    openai_client=openai_client,
    top_k=5
)

# Test the retriever
test_docs = retriever._get_relevant_documents("cozy apartment near downtown")
print(f"\n✅ Retriever test successful")
print(f"📊 Retrieved {len(test_docs)} documents")
print(f"\n📄 First document preview:")
print(test_docs[0].page_content[:400] + "...")

## Step 6: Design the Prompt Template

The prompt template defines the AI's role and behavior. It includes:
- System instructions for the AI assistant
- Placeholders for retrieved context
- Chat history for conversation memory
- The user's question

In [ ]:
# System prompt that defines the AI's role and behavior
system_prompt = """You are a friendly and knowledgeable Airbnb assistant helping users find their perfect accommodation.

Your responsibilities:
- Analyze the user's requirements and preferences
- Recommend suitable listings from the provided context
- Explain why each listing matches their needs
- Be conversational, enthusiastic, and helpful
- If the user asks follow-up questions, use the conversation history to provide context-aware responses

Guidelines:
- Always base your recommendations on the retrieved listings context
- Highlight key features that match the user's stated preferences
- Mention price, location, and standout amenities
- If none of the listings are perfect matches, explain what's close and ask if they want to adjust criteria
- Keep responses concise but informative (2-4 sentences per listing)
- Use a friendly, conversational tone

Context with relevant listings:
{context}

Remember: Only recommend listings that appear in the context above. Do not make up or hallucinate listings."""

# Create the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{question}")
])

print("✅ Prompt template created")

## Step 7: Build the RAG Chain

The RAG chain connects:
1. Retriever → fetches relevant documents
2. Formatter → prepares documents for the prompt
3. Prompt → structures the input for the LLM
4. LLM → generates the response
5. Parser → extracts the text response

In [ ]:
def format_docs(docs: List[Document]) -> str:
    """Format retrieved documents into a readable context string."""
    if not docs:
        return "No relevant listings found."
    
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(f"\n--- Listing {i} ---")
        formatted.append(doc.page_content)
        formatted.append(f"Similarity Score: {doc.metadata.get('similarity_score', 0):.4f}")
    
    return "\n".join(formatted)

# Create the RAG chain
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
        "chat_history": lambda x: []  # We'll add memory in the next step
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain created")

In [ ]:
# Test the RAG chain with a simple query
test_question = "I'm looking for a cozy place in Chicago for a weekend getaway"
response = rag_chain.invoke(test_question)

print(f"\n🔍 Question: {test_question}")
print(f"\n🤖 Response:")
print(response)

## Step 8: Add Conversation Memory

To handle follow-up questions, we need to maintain conversation history. This allows the AI to understand context like "the first one" or "what about parking?"

In [ ]:
class ConversationManager:
    """
    Manages conversation history and context for RAG interactions.
    """
    
    def __init__(self, retriever, llm, prompt):
        self.retriever = retriever
        self.llm = llm
        self.prompt = prompt
        self.conversations: Dict[str, List] = {}  # session_id -> chat history
    
    def get_or_create_session(self, session_id: str) -> List:
        """Get or create a conversation session."""
        if session_id not in self.conversations:
            self.conversations[session_id] = []
        return self.conversations[session_id]
    
    def chat(self, session_id: str, question: str) -> str:
        """
        Process a chat message and return the AI response.
        
        Args:
            session_id: Unique identifier for the conversation session
            question: The user's question
            
        Returns:
            The AI's response
        """
        # Get conversation history
        chat_history = self.get_or_create_session(session_id)
        
        # Retrieve relevant documents
        docs = self.retriever._get_relevant_documents(question)
        context = format_docs(docs)
        
        # Build the chain with history
        chain = self.prompt | self.llm | StrOutputParser()
        
        # Get response
        response = chain.invoke({
            "context": context,
            "question": question,
            "chat_history": chat_history
        })
        
        # Update conversation history
        chat_history.append(HumanMessage(content=question))
        chat_history.append(AIMessage(content=response))
        
        return response
    
    def clear_session(self, session_id: str):
        """Clear a conversation session."""
        if session_id in self.conversations:
            del self.conversations[session_id]
    
    def get_session_count(self, session_id: str) -> int:
        """Get the number of exchanges in a session."""
        if session_id in self.conversations:
            return len(self.conversations[session_id]) // 2
        return 0

# Initialize conversation manager
conversation_manager = ConversationManager(retriever, llm, prompt)

print("✅ Conversation manager initialized")

In [ ]:
# Test conversation with memory
session_id = "user_123"

print("=" * 80)
print("💬 Conversation Test")
print("=" * 80)

# First message
question1 = "I need a place in Chicago with parking"
response1 = conversation_manager.chat(session_id, question1)
print(f"\n👤 User: {question1}")
print(f"\n🤖 Assistant:\n{response1}")

print("\n" + "-" * 80)

In [ ]:
# Follow-up question (tests memory)
question2 = "Does the first one have a kitchen?"
response2 = conversation_manager.chat(session_id, question2)
print(f"\n👤 User: {question2}")
print(f"\n🤖 Assistant:\n{response2}")

print("\n" + "-" * 80)

In [ ]:
# Another follow-up
question3 = "What about the price? Is it under $150?"
response3 = conversation_manager.chat(session_id, question3)
print(f"\n👤 User: {question3}")
print(f"\n🤖 Assistant:\n{response3}")

print("\n" + "=" * 80)
print(f"📊 Conversation length: {conversation_manager.get_session_count(session_id)} exchanges")

## Step 9: Enhance with Query Rephrasing

For better retrieval, we can rephrase follow-up questions to be standalone. This helps the retriever find more relevant documents when the user's question references previous context.

In [ ]:
# Prompt for rephrasing follow-up questions
rephrase_prompt = ChatPromptTemplate.from_messages([
    ("system", """Given a chat history and a follow-up question, rephrase the follow-up question 
to be a standalone question that includes relevant context from the chat history.

If the question is already standalone, return it as is.

Examples:
- Chat history: [User asks about Chicago apartments]
- Follow-up: "Does the first one have parking?"
- Standalone: "Does the Chicago apartment mentioned first have parking?"

- Chat history: [User asks about pet-friendly places]
- Follow-up: "What about the price?"
- Standalone: "What is the price of the pet-friendly listing?"
"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# Create rephrasing chain
rephrase_chain = rephrase_prompt | llm | StrOutputParser()

print("✅ Rephrasing chain created")

In [ ]:
class EnhancedConversationManager(ConversationManager):
    """
    Enhanced conversation manager with query rephrasing for better retrieval.
    """
    
    def __init__(self, retriever, llm, prompt, rephrase_chain):
        super().__init__(retriever, llm, prompt)
        self.rephrase_chain = rephrase_chain
    
    def rephrase_question(self, session_id: str, question: str) -> str:
        """Rephrase a follow-up question to be standalone."""
        chat_history = self.get_or_create_session(session_id)
        
        if not chat_history:
            return question  # No history, return as is
        
        rephrased = self.rephrase_chain.invoke({
            "chat_history": chat_history,
            "question": question
        })
        
        return rephrased
    
    def chat(self, session_id: str, question: str, verbose: bool = False) -> str:
        """
        Process a chat message with query rephrasing.
        
        Args:
            session_id: Unique identifier for the conversation session
            question: The user's question
            verbose: If True, print the rephrased question
            
        Returns:
            The AI's response
        """
        # Get conversation history
        chat_history = self.get_or_create_session(session_id)
        
        # Rephrase the question if there's history
        search_query = question
        if chat_history:
            search_query = self.rephrase_question(session_id, question)
            if verbose and search_query != question:
                print(f"🔄 Rephrased: '{question}' → '{search_query}'")
        
        # Retrieve relevant documents using rephrased query
        docs = self.retriever._get_relevant_documents(search_query)
        context = format_docs(docs)
        
        # Build the chain with history
        chain = self.prompt | self.llm | StrOutputParser()
        
        # Get response
        response = chain.invoke({
            "context": context,
            "question": question,  # Use original question for response
            "chat_history": chat_history
        })
        
        # Update conversation history
        chat_history.append(HumanMessage(content=question))
        chat_history.append(AIMessage(content=response))
        
        return response

# Initialize enhanced conversation manager
enhanced_manager = EnhancedConversationManager(retriever, llm, prompt, rephrase_chain)

print("✅ Enhanced conversation manager initialized")

In [ ]:
# Test enhanced conversation with rephrasing
session_id = "user_456"

print("\n" + "=" * 80)
print("💬 Enhanced Conversation Test (with rephrasing)")
print("=" * 80)

# First message
q1 = "Show me pet-friendly apartments in Denver"
r1 = enhanced_manager.chat(session_id, q1, verbose=True)
print(f"\n👤 User: {q1}")
print(f"\n🤖 Assistant:\n{r1}")

print("\n" + "-" * 80)

In [ ]:
# Follow-up that needs rephrasing
q2 = "Any with a yard?"
r2 = enhanced_manager.chat(session_id, q2, verbose=True)
print(f"\n👤 User: {q2}")
print(f"\n🤖 Assistant:\n{r2}")

print("\n" + "-" * 80)

In [ ]:
# Another follow-up
q3 = "What's the cheapest option?"
r3 = enhanced_manager.chat(session_id, q3, verbose=True)
print(f"\n👤 User: {q3}")
print(f"\n🤖 Assistant:\n{r3}")

## Step 10: Structure Responses for Frontend Integration

For a chat UI, we want structured responses with metadata that can be used by the frontend.

In [ ]:
import json
from datetime import datetime

def chat_with_structured_response(session_id: str, question: str, filters: dict = None):
    """
    Process a chat message and return a structured response for frontend integration.
    
    Args:
        session_id: Unique identifier for the conversation session
        question: The user's question
        filters: Optional filters to apply
        
    Returns:
        dict: Structured response with message, listings, and metadata
    """
    # Get conversation history
    chat_history = enhanced_manager.get_or_create_session(session_id)
    
    # Rephrase if needed
    search_query = question
    if chat_history:
        search_query = enhanced_manager.rephrase_question(session_id, question)
    
    # Retrieve documents
    docs = retriever._get_relevant_documents(search_query)
    context = format_docs(docs)
    
    # Generate response
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({
        "context": context,
        "question": question,
        "chat_history": chat_history
    })
    
    # Update history
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    
    # Build structured response
    structured_response = {
        "session_id": session_id,
        "timestamp": datetime.now().isoformat(),
        "query": {
            "original": question,
            "rephrased": search_query if search_query != question else None
        },
        "response": {
            "message": response,
            "listings": [
                {
                    "id": doc.metadata.get("id"),
                    "name": doc.metadata.get("name"),
                    "price": doc.metadata.get("price"),
                    "property_type": doc.metadata.get("property_type"),
                    "similarity_score": doc.metadata.get("similarity_score")
                }
                for doc in docs
            ]
        },
        "metadata": {
            "documents_retrieved": len(docs),
            "filters_applied": filters,
            "message_count": enhanced_manager.get_session_count(session_id)
        }
    }
    
    return structured_response

print("✅ Structured response function created")

In [ ]:
# Test structured response
test_session = "user_789"
structured_resp = chat_with_structured_response(
    session_id=test_session,
    question="Find me a luxury condo in Boston under $300"
)

print("\n📦 Structured Response:")
print(json.dumps(structured_resp, indent=2, default=str))

## Step 11: Test Various Conversation Scenarios

Let's test the RAG system with different types of queries to see how it handles various scenarios.

In [ ]:
# Test different conversation scenarios
test_scenarios = [
    {
        "name": "Weekend Getaway",
        "questions": [
            "I want a romantic place for a weekend getaway",
            "Does it have a hot tub?",
            "What's the cancellation policy?"
        ]
    },
    {
        "name": "Business Travel",
        "questions": [
            "I need a place with good wifi for remote work",
            "Is there a desk or workspace?",
            "How about the second option?"
        ]
    },
    {
        "name": "Family Vacation",
        "questions": [
            "Looking for a family-friendly house with 3 bedrooms",
            "Is it kid-safe?",
            "What amenities does it have for children?"
        ]
    }
]

print("🧪 Testing RAG Conversation Scenarios\n")
print("=" * 80)

for i, scenario in enumerate(test_scenarios):
    session_id = f"test_scenario_{i}"
    print(f"\n📋 Scenario: {scenario['name']}")
    print("-" * 40)
    
    for question in scenario['questions']:
        response = enhanced_manager.chat(session_id, question, verbose=True)
        print(f"\n👤 User: {question}")
        print(f"🤖 Assistant: {response[:300]}..." if len(response) > 300 else f"🤖 Assistant: {response}")
    
    print("\n" + "=" * 80)

## 🎓 What You've Learned

✅ **RAG Pattern**: Combining retrieval, augmentation, and generation  
✅ **Custom Retrievers**: Building LangChain retrievers for DocumentDB  
✅ **Prompt Engineering**: Designing effective system prompts for AI assistants  
✅ **Conversation Memory**: Maintaining context across multiple turns  
✅ **Query Rephrasing**: Improving retrieval for follow-up questions  
✅ **Structured Responses**: Formatting output for frontend integration  

## 🎉 What's Next?

In **Module 3: Multi-Agent System with LangGraph**, you'll learn how to:
- Design a multi-agent architecture
- Create specialized agents (Search, Filter, Recommendation)
- Implement agent orchestration with LangGraph
- Handle complex workflows with state management
- Build a conversation router that delegates to the right agent